# Bibliometria Geral no Google Colab

Notebook para rodar analises bibliometricas com arquivos CSV, Excel (.xls/.xlsx) e BibTeX (.bib).

- Upload do arquivo bibliografico
- Tabelas principais (autores, citacoes, termos de titulo e resumo)
- Grafo de coautoria e graficos
- Download de todos os resultados em ZIP

In [ ]:
# Clona o repositorio e instala dependencias
import os
import shutil

REPO_URL = "https://github.com/lucaslmfbib/bibliometria.git"
REPO_DIR = "/content/bibliometria"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}
%cd /content/bibliometria
!pip -q install -r requirements.txt

In [ ]:
# Imports
import json
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display
from google.colab import files

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from bibliometria.pipeline import run_bibliometric_analysis

In [ ]:
# Configuracoes da analise (pode ajustar)
TOP_N = 20
CSV_ENCODING = None        # exemplo: 'utf-8'
SHEET_NAME = 0             # numero ou nome da aba no Excel
GERAR_GRAFICOS = True
NETWORK_MAX_AUTHORS = 30
NETWORK_MIN_WEIGHT = 1

In [ ]:
# Upload do arquivo bibliografico
uploaded = files.upload()
if not uploaded:
    raise ValueError("Nenhum arquivo enviado.")

uploaded_name, uploaded_data = next(iter(uploaded.items()))
input_path = Path("/content") / uploaded_name
input_path.write_bytes(uploaded_data)
print(f"Arquivo salvo em: {input_path}")

In [ ]:
# Executa a analise
output_dir = Path("/content/analysis_output")
if output_dir.exists():
    shutil.rmtree(output_dir)

summary = run_bibliometric_analysis(
    input_path=input_path,
    output_dir=output_dir,
    top_n=TOP_N,
    encoding=CSV_ENCODING,
    sheet_name=SHEET_NAME,
    save_plots=GERAR_GRAFICOS,
    network_max_authors=NETWORK_MAX_AUTHORS,
    network_min_weight=NETWORK_MIN_WEIGHT,
)

display(Markdown("## Resumo da pesquisa"))
display(pd.DataFrame(summary.get("research_overview", [])))

print(
    f"Documentos: {summary.get('total_documents')} | "
    f"Periodo: {summary.get('period_start')} - {summary.get('period_end')}"
)

In [ ]:
# Tabelas principais
def show_table(csv_name, title, rows=20, sort_by=None, ascending=False):
    csv_path = output_dir / csv_name
    if not csv_path.exists():
        display(Markdown(f"### {title}"))
        print(f"Arquivo nao encontrado: {csv_name}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    if sort_by and sort_by in df.columns:
        df[sort_by] = pd.to_numeric(df[sort_by], errors="coerce")
        df = df.sort_values(by=sort_by, ascending=ascending, na_position="last")

    display(Markdown(f"### {title}"))
    display(df.head(rows))
    return df

authors_df = show_table(
    "author_document_counts.csv",
    "Autores e numero de trabalhos",
    rows=100,
    sort_by="documents",
    ascending=False,
)

most_cited_df = show_table(
    "most_cited_documents.csv",
    "Documentos mais citados",
    rows=100,
    sort_by="citations",
    ascending=False,
)

title_terms_df = show_table(
    "title_terms.csv",
    "Palavras-chave no titulo",
    rows=50,
    sort_by="count",
    ascending=False,
)

abstract_terms_df = show_table(
    "abstract_terms.csv",
    "Palavras-chave no resumo",
    rows=50,
    sort_by="count",
    ascending=False,
)

In [ ]:
# Grafo de coautoria e lista de graficos
graph_path = output_dir / "coauthorship_network.png"
if graph_path.exists():
    display(Markdown("### Grafo de coautoria"))
    display(Image(filename=str(graph_path)))
else:
    print("Grafo de coautoria nao gerado para este arquivo.")

display(Markdown("### Arquivos gerados"))
for generated_file in sorted(output_dir.iterdir()):
    print(generated_file.name)

In [ ]:
# Download de todos os resultados em ZIP
zip_path = Path("/content/analysis_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(output_dir.rglob("*")):
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.relative_to(output_dir))

print(f"ZIP criado em: {zip_path}")
files.download(str(zip_path))